In [3]:
import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

In [6]:
def build_dataset(seed_url):
    articles = {
        'topic': [],
        'link': [],
        'header': [],   
    }
    data = requests.get(seed_url)
    time.sleep(1) # sleeping for 1 second to make sure we are polite!
    soup = BeautifulSoup(
        data.content.decode('utf-8', 'ignore'),
        'html.parser',
    )
    print(soup)
    candidates = soup.find_all(
        "a",
        class_=[
            "loop-card__title-link", # website specific
        ],
    )
    for article in candidates:
        articles['topic'].append(seed_url.split('/')[-1])
        articles['link'].append(article['href'])
        articles['header'].append(article.get_text().strip())
    
    return pd.DataFrame(articles)

base_url = 'https://www.minecraftskins.com/search/skin/spiderman/1/'

# Crawl and parse the AI page
df = build_dataset(base_url)
print(df)

<!DOCTYPE html>

<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>
<title>Attention Required! | Cloudflare</title>
<meta charset="utf-8"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
<meta content="noindex, nofollow" name="robots"/>
<meta content="width=device-width,initial-scale=1" name="viewport"/>
<link href="/cdn-cgi/styles/cf.errors.css" id="cf_styles-css" rel="stylesheet"/>
<!--[if lt IE 9]><link rel="stylesheet" id='cf_styles-ie-css' href="/cdn-cgi/styles/cf.errors.ie.css" /><![endif]-->
<style>body{margin:0;padding:0}</style>
<!--[if gte IE 10]><!-->
<script>
  if (!navigator.cookieEnabled) {
    window.addEventListener('DOMContentLoaded', function

In [1]:
# <div class="skin-img">
#          <a href="/skin/23964959/spider-man-brand-new-day-suit--no-webshooters-variant-/">

#<div class="embed image-link-code">
#    <label for="image-link-code">Image Link:</label>
#    <input id="image-link-code" type="text" value="https://www.minecraftskins.com/uploads/skins/2026/04/02/spider-man-brand-new-day-suit--no-webshooters-variant--23964959.png?v950" />
#</div>


import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.minecraftskins.com"
SEARCH_URL = "https://www.minecraftskins.com/search/skin/spiderman/1/"

#HEADERS = {
#    "User-Agent": "Mozilla/5.0 (compatible; SkinCrawler/1.0)"
#}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive",
}

OUTPUT_DIR = "skins"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_html(url):
    response = requests.get(url, headers=HEADERS) # , timeout=10)
    response.raise_for_status()
    return response.text


def extract_skin_links(search_html):
    soup = BeautifulSoup(search_html, "html.parser")
    links = []

    for a in soup.find_all("a", href=True):
        if "/skin/" in a["href"]:
            full_url = urljoin(BASE_URL, a["href"])
            links.append(full_url)

    # Duplikate entfernen
    return list(set(links))


def extract_image_url(detail_html):
    soup = BeautifulSoup(detail_html, "html.parser")

    img = soup.find("img", {"src": True})

    if img and "uploads/skins" in img["src"]:
        return img["src"]

    return None


def download_image(url):
    filename = url.split("/")[-1].split("?")[0]
    path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(path):
        print(f"Skip existing: {filename}")
        return

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    with open(path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {filename}")


def run():
    print("Loading search page...")
    html = get_html(SEARCH_URL)

    print("Extracting skin links...")
    skin_links = extract_skin_links(html)
    print(f"Found {len(skin_links)} skins")

    for i, link in enumerate(skin_links, 1):
        try:
            print(f"[{i}/{len(skin_links)}] Processing {link}")

            detail_html = get_html(link)
            image_url = extract_image_url(detail_html)

            if image_url:
                download_image(image_url)
            else:
                print("No image found")

            time.sleep(1)  # wichtig gegen Rate Limit

        except Exception as e:
            print(f"Error: {e}")


if __name__ == "__main__":
    run()

Loading search page...


HTTPError: 403 Client Error: Forbidden for url: https://www.minecraftskins.com/search/skin/spiderman/1/